# Know Your Batik — Exploratory Data Analysis

In [2]:
import os
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image, UnidentifiedImageError

warnings.filterwarnings('ignore')

DATASET_DIR = Path(r'C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset')
OUTPUT_DIR  = Path('../outputs/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
sns.set_theme(style='whitegrid', palette='muted')

## Step 1 — LOAD

In [3]:
records = []
for cls_dir in sorted(DATASET_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    for img_path in cls_dir.iterdir():
        if img_path.suffix.lower() in VALID_EXTS:
            records.append({
                'path':       str(img_path),
                'class_name': cls_dir.name,
                'filename':   img_path.name,
                'extension':  img_path.suffix.lower(),
            })

df = pd.DataFrame(records)
print(f'Total images : {len(df)}')
print(f'Class count  : {df["class_name"].nunique()}')
print(f'Extensions   : {df["extension"].value_counts().to_dict()}')
df.head()

Total images : 2216
Class count  : 28
Extensions   : {'.jpg': 2199, '.jpeg': 9, '.png': 5, '.webp': 3}


,path,class_name,filename,extension
0,C:\Users\fmoch\OneDrive\Documents\Data Kuliah\...,Bali_Barong,Bali_Barong1.jpg,.jpg
1,C:\Users\fmoch\OneDrive\Documents\Data Kuliah\...,Bali_Barong,Bali_Barong10.jpg,.jpg
2,C:\Users\fmoch\OneDrive\Documents\Data Kuliah\...,Bali_Barong,Bali_Barong11.jpg,.jpg
3,C:\Users\fmoch\OneDrive\Documents\Data Kuliah\...,Bali_Barong,Bali_Barong12.jpg,.jpg
4,C:\Users\fmoch\OneDrive\Documents\Data Kuliah\...,Bali_Barong,Bali_Barong13.jpg,.jpg


## Step 2 — CLEAN

In [4]:
# --- corrupt check ---
corrupt_paths = []
for path in df['path']:
    try:
        with Image.open(path) as img:
            img.verify()
    except (UnidentifiedImageError, Exception):
        corrupt_paths.append(path)

df = df[~df['path'].isin(corrupt_paths)].reset_index(drop=True)
print(f'Corrupt images removed : {len(corrupt_paths)}')

# --- duplicate check via MD5 ---
def md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

df['md5'] = df['path'].apply(md5)
dup_mask = df.duplicated(subset='md5', keep='first')
n_duplicates = dup_mask.sum()
df = df[~dup_mask].reset_index(drop=True)

print(f'Duplicate images removed : {n_duplicates}')
print(f'Final dataset size       : {len(df)}')

Corrupt images removed : 0
Duplicate images removed : 89
Final dataset size       : 2127


## Step 3 — STATS

In [5]:
class_counts = df.groupby('class_name').size().rename('count')

stats = pd.DataFrame({
    'count':  class_counts,
    'min':    class_counts.min(),
    'max':    class_counts.max(),
    'mean':   class_counts.mean(),
    'std':    class_counts.std(),
    'median': class_counts.median(),
})

# per-class describe
desc = class_counts.describe().rename({
    'count': 'num_classes', 'mean': 'mean_imgs',
    'std': 'std_imgs', 'min': 'min_imgs',
    '25%': 'q1', '50%': 'median', '75%': 'q3', 'max': 'max_imgs'
})
print('=== Per-class image count statistics ===')
print(desc.to_frame().T.to_string(index=False))
print()

small_classes = class_counts[class_counts < 50]
if len(small_classes):
    print(f'Classes with < 50 images ({len(small_classes)}):')
    print(small_classes.to_string())
else:
    print('All classes have >= 50 images.')

print()
print(class_counts.sort_values(ascending=False).to_frame())

=== Per-class image count statistics ===
 num_classes  mean_imgs  std_imgs  min_imgs   q1  median    q3  max_imgs
        28.0  75.964286 41.322115      34.0 53.0    63.5 76.25     202.0

Classes with < 50 images (5):
class_name
Bali_Merak                49
Ceplok                    48
Ikat_Celup                46
Priangan_Merak_Ngibing    34
Sidoluhur                 48

                             count
class_name                        
Jawa_Barat_Megamendung         202
Kalimantan_Dayak               197
Solo_Parang                    126
Yogyakarta_Kawung              121
Papua_Cendrawasih              116
Yogyakarta_Parang               82
Sumatera_Utara_Boraspati        80
Corak_Insang                    75
Madura_Mataketeran              72
Maluku_Pala                     68
Sulawesi_Selatan_Lontara        68
Papua_Tifa                      68
Papua_Asmat                     67
Sumatera_Barat_Rumah_Minang     65
Lasem                           62
Sekar                         

## Step 4 — VISUALIZE

### 4a. Class Distribution

In [6]:
counts_sorted = class_counts.sort_values(ascending=False)
mean_val = counts_sorted.mean()

fig, ax = plt.subplots(figsize=(12, 9))
bars = ax.barh(counts_sorted.index[::-1], counts_sorted.values[::-1],
               color=sns.color_palette('muted', len(counts_sorted)))
ax.axvline(mean_val, color='crimson', linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_val:.1f}')
ax.set_xlabel('Number of Images', fontsize=12)
ax.set_title('Batik Class Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
for bar, val in zip(bars, counts_sorted.values[::-1]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150)
plt.close(fig)
print('Saved: class_distribution.png')

Saved: class_distribution.png


### 4b. Image Dimensions

In [7]:
dims = []
for path in df['path']:
    try:
        with Image.open(path) as img:
            dims.append(img.size)  # (width, height)
    except Exception:
        dims.append((0, 0))

df['width'], df['height'] = zip(*dims)
df_valid = df[(df['width'] > 0) & (df['height'] > 0)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Image Dimensions Analysis', fontsize=14, fontweight='bold')

axes[0, 0].hist(df_valid['width'],  bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Width Distribution');  axes[0, 0].set_xlabel('Pixels')

axes[0, 1].hist(df_valid['height'], bins=40, color='coral',     edgecolor='white')
axes[0, 1].set_title('Height Distribution'); axes[0, 1].set_xlabel('Pixels')

axes[1, 0].boxplot(df_valid['width'],  vert=True, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1, 0].set_title('Width Boxplot'); axes[1, 0].set_ylabel('Pixels')

axes[1, 1].boxplot(df_valid['height'], vert=True, patch_artist=True,
                   boxprops=dict(facecolor='coral', alpha=0.6))
axes[1, 1].set_title('Height Boxplot'); axes[1, 1].set_ylabel('Pixels')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'image_dimensions.png', dpi=150)
plt.close(fig)
print('Saved: image_dimensions.png')
print(f'Width  — min:{df_valid["width"].min()}  max:{df_valid["width"].max()}  mean:{df_valid["width"].mean():.0f}')
print(f'Height — min:{df_valid["height"].min()}  max:{df_valid["height"].max()}  mean:{df_valid["height"].mean():.0f}')

Saved: image_dimensions.png
Width  — min:100  max:3648  mean:533
Height — min:100  max:3072  mean:513


### 4c. Pixel Intensity Distribution

In [8]:
sample_paths = df['path'].sample(n=min(100, len(df)), random_state=42).tolist()

all_pixels = []
for path in sample_paths:
    try:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            all_pixels.append(img.flatten())
    except Exception:
        pass

all_pixels = np.concatenate(all_pixels)

fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(all_pixels, ax=ax, fill=True, color='mediumslateblue', alpha=0.6)
ax.set_xlim(0, 255)
ax.set_xlabel('Pixel Intensity', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Pixel Intensity Distribution (100 random grayscale images)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'pixel_distribution.png', dpi=150)
plt.close(fig)
print('Saved: pixel_distribution.png')

Saved: pixel_distribution.png


### 4d. Sample Grid (1 image per class)

In [9]:
classes = sorted(df['class_name'].unique())
n_cols, n_rows = 7, 4

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 12))
fig.suptitle('Sample Images per Batik Class', fontsize=16, fontweight='bold', y=1.01)

for idx, cls in enumerate(classes):
    ax = axes[idx // n_cols][idx % n_cols]
    sample = df[df['class_name'] == cls].sample(1, random_state=42).iloc[0]
    try:
        with Image.open(sample['path']) as img:
            img_rgb = img.convert('RGB').resize((224, 224))
        ax.imshow(np.array(img_rgb))
    except Exception:
        ax.set_facecolor('#cccccc')
    ax.set_title(cls.replace('_', ' '), fontsize=7, pad=3)
    ax.axis('off')

# hide unused axes (28 classes fills 7x4 exactly)
for idx in range(len(classes), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].axis('off')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'sample_grid.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved: sample_grid.png')

Saved: sample_grid.png


## Step 5 — OUTLIERS

In [10]:
Q1 = df['width'].quantile(0.25)
Q3 = df['width'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df['width'] < lower) | (df['width'] > upper)]
print(f'IQR bounds : [{lower:.0f}, {upper:.0f}]')
print(f'Width outliers found : {len(outliers)}')

if len(outliers):
    print('\nOutlier paths:')
    for _, row in outliers.iterrows():
        print(f'  [{row["class_name"]}]  w={row["width"]}  h={row["height"]}  {row["path"]}')
    print()
    print('Decision: KEEP — all images will be resized to 224x224 during preprocessing.')
    print('Extreme dimensions are not a quality problem; resizing handles them uniformly.')
else:
    print('No width outliers — all images within IQR bounds.')

IQR bounds : [-340, 1164]
Width outliers found : 190

Outlier paths:
  [Bali_Barong]  w=1200  h=1200  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Barong\Bali_Barong10.jpg
  [Bali_Barong]  w=1281  h=1281  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Barong\Bali_Barong4.jpg
  [Bali_Merak]  w=1719  h=1719  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Merak\Bali5.jpg
  [Bali_Merak]  w=1172  h=1172  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Merak\Bali_Merak11.jpg
  [Bali_Merak]  w=1365  h=1365  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Merak\Bali_Merak18.jpg
  [Bali_Merak]  w=1600  h=1600  C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\Know Your Batik\dataset\Bali_Me

## Step 6 — SUMMARY

In [11]:
total       = len(df)
n_classes   = df['class_name'].nunique()
counts      = df.groupby('class_name').size()
small_cls   = counts[counts < 50]
imb_ratio   = counts.max() / counts.min()
med_w       = int(df['width'].median())
med_h       = int(df['height'].median())

print('=' * 52)
print('         KNOW YOUR BATIK — EDA SUMMARY')
print('=' * 52)
print(f'Total images            : {total}')
print(f'Total classes           : {n_classes}')
print(f'Min images per class    : {counts.min()}  ({counts.idxmin()})')
print(f'Max images per class    : {counts.max()}  ({counts.idxmax()})')
print(f'Mean images per class   : {counts.mean():.1f}')
print(f'Classes with < 50 imgs  : {len(small_cls)} — {list(small_cls.index) if len(small_cls) else "None"}')
print(f'Typical image size      : ~{med_w} x {med_h} px (median)')
print(f'Recommended resize      : 224 x 224 px  (ImageNet standard)')
print(f'Imbalance ratio         : {imb_ratio:.2f}x  (max / min class)')
if imb_ratio > 2:
    print('Recommended strategy    : Weighted loss or oversampling (imbalance > 2x)')
else:
    print('Recommended strategy    : Standard training (imbalance acceptable)')
print('=' * 52)

         KNOW YOUR BATIK — EDA SUMMARY
Total images            : 2127
Total classes           : 28
Min images per class    : 34  (Priangan_Merak_Ngibing)
Max images per class    : 202  (Jawa_Barat_Megamendung)
Mean images per class   : 76.0
Classes with < 50 imgs  : 5 — ['Bali_Merak', 'Ceplok', 'Ikat_Celup', 'Priangan_Merak_Ngibing', 'Sidoluhur']
Typical image size      : ~291 x 276 px (median)
Recommended resize      : 224 x 224 px  (ImageNet standard)
Imbalance ratio         : 5.94x  (max / min class)
Recommended strategy    : Weighted loss or oversampling (imbalance > 2x)
